# Gépi Fordítás (50 pont)




**A notebookot importáld be a Colab rendszerbe, majd abban dolgozz!**

**Versenyző neve: [ÍRD IDE A NEVED]**


Stux, de helyette most Intelligens János vérbeli párizsi lett. A francia körútja során francia nyelvű kifejezéseket, mondatokat és különféle nyelvű személyneveket gyűjtött a jegyzetfüzetébe.
Mivel a francia nyelvet nem sikerült még teljesen elsajátítania, így egy számítógépes fordítót hív segítségül azért, hogy a feljegyzéseket megérthesse. Készíts neki egy ilyen programot!

---
## Előkészítések

**Útmutatók a feladat megoldásához szükséges eszközökhöz:**
1. [Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
2. [Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
3. [Tokenization](https://medium.com/@utkarsh.kant/tokenization-a-complete-guide-3f2dd56c0682)
4. [Tokenization, Mapping and Padding](https://medium.com/@lokaregns/preparing-text-data-for-transformers-tokenization-mapping-and-padding-9fbfbce28028)
5. [PyTorch Training/Inference](https://pytorch.org/tutorials/beginner/introyt/trainingyt.html)
6. [PyTorch Datasets and DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
7. [Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
8. [Matplotlib Bar Plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html)
9. [LSTM, GRU](https://medium.com/@mervebdurna/nlp-with-deep-learning-neural-networks-rnns-lstms-and-gru-3de7289bb4f8)
10. [GRU PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html)
11. [Next Word Prediction](https://www.geeksforgeeks.org/next-word-prediction-with-deep-learning-in-nlp/)



A megoldáshoz szükséges **modell súlyok** a következő linken érhetőek el: [Encoder](https://drive.google.com/file/d/16p4Ky4RyD8o_6wCp0VV3KukVmmOUPvV4/view?usp=sharing), [Decoder](https://drive.google.com/file/d/1vmBeAjTr7mK4mI-8h7UkkII5NWSd2TT9/view?usp=sharing)

In [ ]:
# @title Függőségek Telepítése
!pip install pandas --quiet
!pip install torchtext --quiet

In [ ]:
# @title Nyelvi Adatahalmaz és a Modell Súlyok letöltése
!gdown -qq 16p4Ky4RyD8o_6wCp0VV3KukVmmOUPvV4
!gdown -qq 1vmBeAjTr7mK4mI-8h7UkkII5NWSd2TT9
!gdown -qq 16O8wQ0-LYrANbYVuFGnX7epiIB2zIqcY
!unzip -qq nyelvek.zip -d .
!rm -rf nyelvek.zip

### Szükséges Könyvtárak

Importáltunk néhány könyvtárat a kezdéshez, de nyugodtan használhatsz bármilyen PyTorch-alapú eszközt, ha szükséges. Kérjük, vedd figyelembe, hogy a Keras és a TensorFlow **NEM ENGEDÉLYEZETT** ennek a feladatnak a megoldásához!

In [ ]:
import io
import re
import math
import random
import unicodedata

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

from tqdm.notebook import tqdm
from sklearn.utils import shuffle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1\. Feladat: Nyelvi Reprezentáció (10 Pont)

Az első feladatban az adathalmazzal való ismerkedést és egy nyelvi reprezentáció elkészítését szeretnénk elérni. A feladataid a következők:

#### 1. Írasd ki az adathalmaz (.txt file) első 10 mondat párosítását! [Pandas Read Text File](https://www.geeksforgeeks.org/how-to-read-text-files-with-pandas/) (2 pont)

#### 2. Készíts egy **Nyelv** osztályt, melynek célja egy nyelv szókincsének kezelése és a szöveges adatoknak a gépi tanulási modellekhez szükséges **numerikus formátumokba** történő átalakítása! (4 pont)

 A Nyelv osztály felépítése a következő legyen:

- **Attribútumok**:
  - `name`: A nyelv nevét jelölő karakterlánc.
  - `word2index`: Egy szótár, amely a szavakat az egyedi indexeikhez rendeli.
  - `word2count`: Szótár az egyes szavak előfordulásának megszámlálásához.
  - `index2word`: Egy szótár, amely indexeket rendel a szavakhoz, előre inicializálva az „SOS” (mondat eleje), „EOS” (mondat vége) és „PAD” (kitöltés) tokenekkel.
  - `n_words`: Az egyedi szavak egész számú reprezentációja, **3-tól kezdődően**, a speciális (SOS, EOS, PAD) tokenek figyelembevétele érdekében.
- **Módszerek**:
  - `addSentence(sentence)`: A mondatot szavakra bontja, és az egyes szavakat az `addWord` segítségével dolgozza fel.
  - `addWord(word)`: Hozzáad egy szót a szótárakhoz, frissítve az összes releváns attribútumot.

#### 3. Az osztály meghatározása mellett a következő segítő függvényekre is szükségünk lesz. Ezek közül kettőt segítségképpen már megírtunk nektek. Implementáljátok a `read_langs` függvényt! (4 pont)

- `unicodeToAscii(s)`: Egy Unicode karakterlánc átalakítása egyszerű ASCII karakterlánccá. **(ADOTT)**
  
- `normalizeString(s)`: A karakterláncadatok előkészítése a feldolgozásra alkalmas formátum megtisztításával és szabványosításával. **(ADOTT)**

- `read_langs(lang1, lang2, reverse)`
    - **Paraméterek**:
      - `lang1`, `lang2`: Az adatállományban szereplő két nyelv neve.
      - `reverse`: Boolean, amely jelzi, hogy a feldolgozás során megforduljon-e a nyelvpárok sorrendje.
    - **Funkcionalitás**:
      - Egy megadott fájl beolvasása, amely két nyelv párosított mondatait tartalmazza.
      - A mondatok normalizálása a `normalizeString` használatával.
      - A `Lang` osztály példányainak létrehozása a szókincskezeléshez.
      - Opcionálisan megfordítja a nyelvpárokat, ha paraméterként meg van adva.

In [ ]:
class Lang():
    def __init__(self, name):
        raise NotImplementedError("This function has not been implemented yet.")

    def add_sentence(self, sentence):
        raise NotImplementedError("This function has not been implemented yet.")

    def add_word(self, word):
        raise NotImplementedError("This function has not been implemented yet.")


def unicode_to_ascii(s):
  return ''.join(
      c for c in unicodedata.normalize('NFD', s)
      if unicodedata.category(c) != 'Mn')

def normalize_string(s):
  s = unicode_to_ascii(s.lower().strip())
  s = re.sub(r"([.!?])", r" \1", s)
  s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
  return s


def read_langs(lang1, lang2, reverse=False):
    raise NotImplementedError("This function has not been implemented yet.")

# 2\. Feladat: Adattisztítás (15 pont)

Az adattisztitás során gyakori társalgási kifejezéseket reprezentáló mondatokat szeretnénk kiszűrni. Erre segítségképp készítettunk egy `eng_prefixes` változót, ami tartalmazza azokat a kezdő mondatrészeket, melyek alapján szűrni szeretnénk a teljes adathalmazt.

#### 1. Példányosíts egy francia és egy angol Nyelvet a `read_langs` függvény segítségével, a nyelvpárok megfordításával! (francia-angol) (3 pont)

#### 2. Szűrd ki az angol nyelvből azokat a mondatokat, melyek nem a `eng_prefixes` **tuple-ben** található szavakkal kezdődnek! (3 pont)

#### 3. Szűrd ki azokat a mondatokat is az angol nyelvből, ahol a mondatok szavainak számossága meghaladja a 10-et! (`MAX_LENGTH`) (3 pont)

#### 4. A két példányosított nyelv attribútumait frissítsd a szűrt mondatok és az `add_senteces` függvény segítségével! (3 pont)

#### 5. Implementáld a `prepare_data` függvényt, ami a két nyelvi objektumot és a megszűrt mondatpárokat fogja visszaadni egy nagy listában! (3 pont)

In [ ]:
MAX_LENGTH = 10
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)


def filter_pairs():
    raise NotImplementedError("This function has not been implemented yet.")


def prepare_data():
    raise NotImplementedError("This function has not been implemented yet.")

# 3\. Feladat: Szavak Eloszlása (6 pont)

Az adattisztítás során gyakori társalgási kifejezéseket reprezentáló mondatokat szeretnénk kiszűrni. Erre segítségképp készítettunk egy `eng_prefixes` változót, ami tartalmazza azokat a kezdő mondatrészeket, melyek alapján szűrni szeretnénk a teljes adathalmazt.

#### 1. Számold ki, hogy a szavak hány százaléka teszi ki az össz szövegállomány 80%-át az angol és a francia nyelv esetén is! (3 pont)

#### 2. Készíts két Bar Plot-ot, ami vizualizálja a 100 legtöbbet használt angol és legtöbbet használt francia szót! (3 pont) [Matplotlib Bar Plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html)

In [ ]:
def plot_lang():
    raise NotImplementedError("This function has not been implemented yet.")

# 4\. Feladat: RNN Architektúra (12 pont)

Célunk egy olyan hálózat létrehozása, amely egy bemeneti mondatot fogad egy nyelven, majd a mondat fordítását adja meg egy kimeneti nyelven.  A hálózatunk egy RNN-t fog használni, amely egy kódolóból és egy dekódolóból áll. A kódoló először a bemeneti mondatunkat vektorrá alakítja, és ezt a sűrített vektort átadja a dekódernek, amely lefordítja a szöveget az adott nyelvre. Ennek folyamatát az alábbi ábra mutatja be részletesebben:

<img src="https://raw.githubusercontent.com/NeuromatchAcademy/course-content-dl/main/projects/static/seq2seq.png" width="600" height="300">

[LSTM, GRU](https://medium.com/@mervebdurna/nlp-with-deep-learning-neural-networks-rnns-lstms-and-gru-3de7289bb4f8)

Azt szeretnénk, hogy egy olyan modellt készítsünk, ami francia nyelvből angol fordítást tud generálni számunkra. Segítségképp a dekódolónak az architektúráját elkészítettük, így a te feladatod a kódoló architektúrájnak a megépítése lesz. A feladataid a következők:

#### 1. Hozd létre a kódoló architektúráját az ábra alapján! (4 pont) [Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)

#### 2. Példányosítsd a két modellt, majd töltsd be a kódoló és a dekódoló súlyait (`decoder.pth`, `encoder.pth`)! (1 pont) ([Pytorch Loading Weights](https://pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html#:~:text=To%20load%20model%20weights%2C%20you,parameters%20using%20load_state_dict()%20method.&text=be%20sure%20to%20call%20model,normalization%20layers%20to%20evaluation%20mode))

#### 3. Konvertáld a nyelvi szekvenciákat megfelelő formátumra a következőképpen: (7 pont)

  * Minden egyes bemeneti mondat esetében alakítsuk át a szavakat a megfelelő indexükre, illesszünk be egy kezdő jelzőt (`SOS`), egy végjelzőt (`EOS`), és biztosítsuk, hogy a szekvencia elé elegendő padding tokent (`PAD`) illesszünk ahhoz, hogy az egységes hosszúság megmaradjon (`MAX_LENGTH`)! A bemeneti mondatot bal oldali kitöltéssel ábrázoljuk, mivel a GRU-hálózat a mondatot balról jobbra haladva dolgozza fel, és azt szeretnénk, ha a kimenet a lehető legközelebb állna a mondatunkhoz. [GRU PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html)
  * Minden egyes kimeneti mondat esetében kezdjünk egy kezdő tokennel (`SOS`), amit követünk az egyes szavak indexeivel! Illesszünk hozzá egy vég tokent (`EOS`), és zárjuk le padding tokenekkel (`PAD`) `MAX_LENGTH`-ig! A GRU-val ellentétben a részleges fordítási mondathoz jobb oldali kitöltést használjunk, mert azt szeretnénk, hogy a kontextusunkat a dekóderünk azonnal feldolgozza. Végül a cél szekvenciánk a részleges fordításunk eggyel való eltolása balra (`next word prediction`, [Next Word Prediction](https://www.geeksforgeeks.org/next-word-prediction-with-deep-learning-in-nlp/)).

  * A célszekvenciát a kimeneti szekvenciából kell előállítani a kezdő token `SOS` eltávolításával és a szekvencia eltolásával, hogy a modell a következő szót tudja megjósolni a szekvenciában. A hossz megtartása érdekében a végére illesszünk `PAD` tokeneket `MAX_LENGTH`-ig!

Kódoló architektúrája:

<a href="https://ibb.co/LNgtxX6"><img src="https://i.ibb.co/vXdP36m/encoder.png" alt="encoder" border="0"></a>

Dekódoló architektúrája:


<a href="https://ibb.co/1qWysVj"><img src="https://i.ibb.co/wWmGJHj/decoder.png" alt="decoder" border="0"></a>

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        raise NotImplementedError("This function has not been implemented yet.")

    def forward(self, input, hidden):
        raise NotImplementedError("This function has not been implemented yet.")

    def initHidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_size, device=device)

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        raise NotImplementedError("This function has not been implemented yet.")

    def forward(self, input, hidden):
        raise NotImplementedError("This function has not been implemented yet.")

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [ ]:
def prepare_data(input_lang, output_lang, pairs, max_len=MAX_LENGTH+2):
    x_input = []
    x_output = []
    target = []
    return x_input, x_output, target

# 5\. Feladat: Fordítások Generálása (7 pont)

Utolsó feladatként szeretnénk tesztelni a modellünk képességeit. A feladataid a következők:

#### 1. A `predict` függvény felhasználásával generálj tokeneket! (1 pont)

#### 2. Fordítsd le a francia-angol mondatpár listából az első 10 francia mondatot angolra. (6 pont)

Egy teljes szöveg generálás egy nagyon egyszerű iteratív folyamat:

* A kimeneti mondat inicializálása csak egy `SOS` tokent jelent, majd a következő iteratív folyamatot szükséges követni:
    1. Jósold meg a következő token valószínűségi eloszlását a kezdetleges kimeneti mondat alapján! (`Softmax`, architektúrában megjelenő)
    2. Válaszd ki a legvalószínűbb tokent!
    3. Add hozzá ezt a tokent a kimeneti mondathoz!
    4. Ha a kapott token `EOS` zárd be a loop-ot!

In [ ]:
def predict(encoder, decoder, input, output):
  _, hidden = encoder(input, encoder.initHidden(input.shape[0]))
  out, _ = decoder(output, hidden)
  return out

In [ ]:
def gen_translation(encoder, decoder, text, input_lang, output_lang, max_len=MAX_LENGTH+2):
    raise NotImplementedError("This function has not been implemented yet.")

def predict_first_ten():
    raise NotImplementedError("This function has not been implemented yet.")

---
**A feladat végére értél**. A kész Notebook-ot töltsd le a következő módon:
```
File → Download → Download .ipynb
```
majd a **többi megoldással együtt** töltsd fel a CMS rendszerbe becsomagolva **(.zip)**.